# 21. Generative Models

forward noising과 noise 예측 목적, flow의 Euler integration을 계산합니다.

핵심 실습은 NumPy와 Python 표준 라이브러리만 사용합니다. API 호출, 모델 다운로드, GPU는 사용하지 않습니다. 수치는 고정된 작은 입력의 계산이며 실제 pretrained model의 성능이 아닙니다.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(27)

## 알려진 corruption

훈련 시에는 clean 값과 추가한 noise를 알기 때문에 회귀 target을 만들 수 있습니다.

In [2]:
x0=np.array([2.,-1.,.5])
epsilon=np.array([.5,-.2,1.])
signal,noise=.6,.8
xt=signal*x0+noise*epsilon
print("noisy sample:",xt)
assert np.isclose(signal**2+noise**2,1)
assert np.isclose(xt[0],1.6)
# Oracle recovery illustrates the algebra, not a trained denoiser.
recovered=(xt-noise*epsilon)/signal
assert np.allclose(recovered,x0)

noisy sample: [ 1.6  -0.76  1.1 ]


## 틀린 noise 예측

oracle과 달리 학습된 denoiser는 오차가 있습니다. 아래는 그 오차의 영향을 직접 계산합니다.

In [3]:
predicted_noise=epsilon+np.array([.1,-.1,.2])
noise_mse=np.square(predicted_noise-epsilon).mean()
reconstructed=(xt-noise*predicted_noise)/signal
print("noise MSE:",noise_mse)
print("reconstruction:",reconstructed)
assert not np.allclose(reconstructed,x0)

noise MSE: 0.019999999999999993
reconstruction: [ 1.8667 -0.8667  0.2333]


## Flow matching의 작은 경로

선형 경로의 target velocity는 두 끝점의 차이입니다. v(x,t)=-x라는 별도 장을 Euler로 적분해 solver step 오차도 확인합니다.

In [4]:
start,end=-1.,2.
t=np.linspace(0,1,5)
path=(1-t)*start+t*end
velocity=end-start
print("linear path:",path,"velocity:",velocity)
assert np.allclose(np.diff(path)/np.diff(t),velocity)
def euler(steps):
    x=1.;dt=1/steps
    for _ in range(steps):x+=dt*(-x)
    return x
print("steps and errors:",[(n,abs(euler(n)-np.exp(-1)))for n in [2,10,100]])
assert abs(euler(100)-np.exp(-1))<abs(euler(2)-np.exp(-1))

linear path: [-1.   -0.25  0.5   1.25  2.  ] velocity: 3.0
steps and errors: [(2, np.float64(0.11787944117144233)), (10, np.float64(0.019201001071442347)), (100, np.float64(0.0018470998982127451))]


## Sampling

공통 HTML demo와 같은 logits를 사용합니다.

In [5]:
logits=np.array([1.,.2,-.4,-1.])
def probabilities(temperature):
    z=logits/temperature;z-=z.max()
    p=np.exp(z);return p/p.sum()
for temperature in [.2,1.,5.]:
    p=probabilities(temperature)
    print(temperature,p)
    assert np.isclose(p.sum(),1)

0.2 [0.9811 0.018  0.0009 0.    ]
1.0 [0.5461 0.2454 0.1347 0.0739]
5.0 [0.305  0.2599 0.2305 0.2045]


## 과제

- signal 계수를 0.1로 줄이고 noise 계수를 제곱합이1이 되게 정하세요. 같은 noise 예측 오차가 복원오차에 미치는 영향을 비교하세요.
- 동일 seed로 sampling한 결과가 반복되는지 확인하고, 다른 seed에서는 결과가 달라질 수 있음을 보이세요.

In [6]:
# 과제 풀이 1
small_signal=.1;large_noise=np.sqrt(1-small_signal**2)
small_xt=small_signal*x0+large_noise*epsilon
small_reconstructed=(small_xt-large_noise*predicted_noise)/small_signal
assert np.linalg.norm(small_reconstructed-x0)>np.linalg.norm(reconstructed-x0)
print("low-signal reconstruction error:",np.linalg.norm(small_reconstructed-x0))
# 과제 풀이 2
p=probabilities(1.)
a=np.random.default_rng(4).choice(4,30,p=p)
b=np.random.default_rng(4).choice(4,30,p=p)
c=np.random.default_rng(5).choice(4,30,p=p)
assert np.array_equal(a,b) and not np.array_equal(a,c)

low-signal reconstruction error: 2.4372115213907883


## 해석

이 실습은 diffusion model을 훈련하지 않습니다. 생성모델의 학습 target·sampling·수치적분을 분리하여 확인합니다. 이미지 품질이나 물리 정확도를 이 결과로 주장할 수 없습니다. [DDPM](https://arxiv.org/abs/2006.11239), [Flow matching](https://arxiv.org/abs/2210.02747).

## Full Model Lab
The lecture HTML contains an inline numerical demo and a Model Lab panel. These notebooks run the corresponding pretrained models in a separate Python environment.
- [21_multimodal-III.ipynb](model-labs/21_multimodal-III.ipynb)
